<a href="https://colab.research.google.com/github/Shahriyar799/Deep_Learning/blob/main/ResNet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!curl -L -o fruits360.zip https://www.kaggle.com/api/v1/datasets/download/moltean/fruits


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 5579M  100 5579M    0     0   117M      0  0:00:47  0:00:47 --:--:--  125M


In [ ]:
!unzip fruits360.zip

Streaming output truncated to the last 5000 lines.
  inflating: fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 3/r0_17.jpg  
  inflating: fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 3/r0_173.jpg  
  inflating: fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 3/r0_177.jpg  
  inflating: fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 3/r0_181.jpg  
  inflating: fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 3/r0_185.jpg  
  inflating: fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 3/r0_189.jpg  
  inflating: fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 3/r0_193.jpg  
  inflating: fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 3/r0_197.jpg  
  inflating: fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 3/r0_201.jpg  
  inflating: fruits-360_original-size/fruits-360-original-s

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

model = models.resnet50(pretrained=True)
num_classes = 10
model.fc = nn.Linear(model.fc.in_features, num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 120MB/s]


In [ ]:
import torch
import torch.nn as nn

# 🔹 Bottleneck blokunun tərifi
class Bottleneck(nn.Module):
    expansion = 4  # Çıxış kanallarını genişləndirmə əmsalı

    def __init__(self, in_channels, mid_channels, stride=1):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_channels)
        self.conv2 = nn.Conv2d(mid_channels, mid_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_channels)
        self.conv3 = nn.Conv2d(mid_channels, mid_channels * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(mid_channels * self.expansion)
        self.relu = nn.ReLU(inplace=True)

        # Shortcut (downsample) bağlantısı
        self.downsample = None
        if stride != 1 or in_channels != mid_channels * self.expansion:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, mid_channels * self.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(mid_channels * self.expansion)
            )

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return self.relu(out)


# 🔹 Tam ResNet‑50 arxitekturası
class ResNet50(nn.Module):
    def __init__(self, num_classes=1000):
        super(ResNet50, self).__init__()
        # Başlanğıc qat (stem)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # Residual blok qrupları
        self.layer1 = self._make_layer(64, 64, 3)          # conv2_x
        self.layer2 = self._make_layer(256, 128, 4, stride=2)  # conv3_x
        self.layer3 = self._make_layer(512, 256, 6, stride=2)  # conv4_x
        self.layer4 = self._make_layer(1024, 512, 3, stride=2) # conv5_x

        # Son qatlar
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(2048, num_classes)

    def _make_layer(self, in_channels, mid_channels, blocks, stride=1):
        layers = []
        layers.append(Bottleneck(in_channels, mid_channels, stride))
        for _ in range(1, blocks):
            layers.append(Bottleneck(mid_channels * Bottleneck.expansion, mid_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


# 🔹 Test
x = torch.randn(1, 3, 224, 224)
model = ResNet50(num_classes=1000)
out = model(x)
print(out.shape)  # (1, 1000)


torch.Size([1, 1000])


In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                     [0.229, 0.224, 0.225])
])

In [ ]:
train_dataset = datasets.ImageFolder(
    root = '/content/fruits-360_100x100/fruits-360/Training',
    transform = transform
)

In [ ]:
class_names = train_dataset.classes

In [ ]:
train_loader = DataLoader(
    train_dataset, batch_size = 32, shuffle = True
)

In [ ]:
weights = models.ResNet50_Weights.IMAGENET1K_V1
resnet50 = models.resnet50(weights=weights)

In [ ]:
resnet50 = resnet50.to(device)

In [ ]:
[key for key, value in resnet50.named_children()]

['conv1',
 'bn1',
 'relu',
 'maxpool',
 'layer1',
 'layer2',
 'layer3',
 'layer4',
 'avgpool',
 'fc']

In [ ]:
# resnet50.fc= resnet50.fc.to(device)

In [ ]:
resnet50.fc = nn.Linear(in_features=2048, out_features=len(class_names), bias = True).to(device)

In [ ]:
len(class_names)

257

In [ ]:
for param in resnet50.parameters():
  param.requires_grad = False

for param in resnet50.fc.parameters():
  param.requires_grad = True

In [ ]:
optimizer = torch.optim.AdamW(resnet50.parameters(), lr=1e-4)
xentropy = nn.CrossEntropyLoss()
n_epoch = 2

In [ ]:
def train(model,optimizer,  criterion,train_loader, n_epoch=10):
  for epoch in range(n_epoch):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
      images = images.to(device)
      labels = labels.to(device)
      pred = model(images)
      loss = criterion(pred, labels)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    print(f'Epoch: {epoch+1}, Loss: {total_loss/len(train_loader)}')


In [ ]:
train(resnet50, optimizer, xentropy, train_loader, n_epoch)

Epoch: 1, Loss: 1.1970995511331002
Epoch: 2, Loss: 0.13976613224678852


In [ ]:
!pip install fastapi uvicorn pyngrok nest-asyncio pillow -q

In [ ]:
import requests

In [ ]:
# from fastapi import FastAPI, UploadFile, File
# from fastapi.middleware.cors import CORSMiddleware
# from PIL import Image
# import io, torch, uvicorn, threading
# import nest_asyncio

# from pyngrok import ngrok
# from torchvision import transforms

# nest_asyncio.apply()

# app = FastAPI()


# app.add_middleware(
#     CORSMiddleware,
#     allow_origins=["*"],
#     allow_methods=["*"],
#     allow_headers=["*"],
# )

# val_transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406],
#                          [0.229, 0.224, 0.225]),
# ])

# @app.post("/predict")
# async def predict(file: UploadFile = File(...)):
#     img = Image.open(io.BytesIO(await file.read())).convert("RGB")
#     tensor = transform(img).unsqueeze(0).to(device)

#     resnet50.eval()
#     with torch.no_grad():
#         probs = torch.softmax(resnet50(tensor), dim=1)[0]

#     top5 = probs.topk(5)

#     return {
#         "predictions": [
#             {"class": class_names[i], "confidence": float(p)}
#             for p, i in zip(top5.values, top5.indices)
#         ]
#     }

In [ ]:
# import glob

# img_path = glob.glob('/content/fruits-360_100x100/fruits-360/Test/*/*.jpg')[0]
# print(img_path)

/content/fruits-360_100x100/fruits-360/Test/Plum 3/r_182_100.jpg


In [ ]:
# with open(img_path, "rb") as f:
#     r = requests.post(
#         f"{public_url}/predict",
#         files={"file": ("image.jpg", f, "image/jpeg")}
#     )
#     print(r.json())

NameError: name 'public_url' is not defined